# 7 - IP Tabulate RCA Valley Bottom Widths

Use this tool to calculate the mean width of RCA valley bottom polygons, and output the results to a table.

## Required Software:

- The code contained in this notebook is designed to be run within an ESRI ArcPro project. The script output is written to the default project geodatabase and therefore the user must open and run the notebook .IPYNB file within an ArcPro project.
- If any of the geoprocessing steps require an advanced license or any specific extensions, the script will check for these conditions before running.

## Required Inputs:

- A geodatabase containing RCA valley bottom polygon feature classes. The feature class names must have a "clean" suffix. These should be the output from the "IP 6 Clip Valley Bottom Polygons by RCA and Clean" tool. Using this tool should result in the appropriate feature class names.
- A field containing unique RCA IDs in the RCA VB feature classes.
- A system directory to use for temporary files.


## Geoprocessing Output:

- A collection of tables containing RCA VB mean valley bottom widths, named using the original RCA VB feature class prefix and written to the same geodatabase containing the RCA VB feature classes.

## Processing Steps:

1. Use the user-provided geodatabase to create a suffix for table output. Optionally, create a list of all valley bottom polygons that have a "clean" suffix.
2. Or, load the clean RCA valley bottoms directly.
3. Define the valley bottom width calculation function. 
4. For each RCA VB feature class, create a list of OIDs and RCAs to use in the function. Also create an empty dataframe to hold function results. Supply all three of these parameters to the function and run it for each RCA VB feature class.
5. Write the populated dataframe to CSV, and then import that CSV to the user-provided geodatabase before moving onto the next RCA VB feature class.
6. Delete any intermediary outputs.

### Code starts here:

#### Setup

Import modules and reset environments to default. This should set the ArcPro project geodatabase as the workspace/scratch environment, just in case it was set otherwise. Additionally, allow the addition of intermediary outputs to the ArcPro project map. (Some selection procedures do not work as expected if the feature classes are not loaded into the map.)

In [ ]:
import arcpy
import pandas
import statistics
from datetime import datetime

arcpy.ResetEnvironments()
arcpy.env.addOutputsToMap = True

User provides filepaths to the geodatabase, the field in the RCA feature class that contains the unique reach ID (usually "RCA_ID" in IP toolbox script outputs), and a system directory for temporary files.

(These inputs are provided directly as text when running the code block from the notebook. Use the hashed out code block below if saving the notebook as a .PY file and creating a tool for use in the ArcPro GUI. In that case, the inputs will be provided as text parameters in "point-and-click" fashion when setting up the tool in the ArcPro GUI.)

In [ ]:
gdb = "F:\\GIS\\IP\\Yukon_Upper_2.gdb"
rid = "RCA_ID"
temp = "C:\\temp"
vb_clean = "RCA_VB_19070502_clean"

In [ ]:
#gdb = arcpy.GetParameterAsText(0)
#rid = arcpy.GetParameterAsText(1)
#temp = arcpy.GetParameterAsText(2)
#vb_clean = arcpy.GetParameterAsText(3)

Define the geodatabase workspace, and use the geodatabase name to establish a prefix for saving the output table.

In [ ]:
arcpy.env.workspace = gdb
g = gdb.split("\\")[-1].split(".gdb")[0]

**Option 1:** list the feature classes in the user-provided geodatabase, and filter the list using suffixes to contain just clean RCA valley bottoms. Only use this option for small geodatabases or drainages, as the processing will slow over time.

If there are many valley bottom polygons to process, load them individually and create a geoprocessing schedule to run each one after the other as a separate script. This will actually go faster, as each time the script ends it will clear temporary system and geodatabase files and improve the overall speed.

In [ ]:
featureclasses = arcpy.ListFeatureClasses()

RCA_VBs = []

for fc in featureclasses:
    if fc.endswith("_clean") == True:
        
        desc = arcpy.Describe(fc)
        fc_full = str(desc.path + "/" + fc)
        
        RCA_VBs.append(fc_full)
    
    else:
        pass

print(RCA_VBs)

**Option 2:** Define the clean valley bottom polygon individually instead.

In [ ]:
RCA_VBs = []
RCA_VBs.append(vb_clean)

Define the valley bottom width calculation function. This function uses minimum bounding geometry to draw a rectangle around each RCA VB polygon on its longest axis, then splits that rectangle into 10 equal widths. The lines intersecting the RCA VB polygon edges are measured and averaged, and the results written to a dataframe.

Arguments to the function are a RCA VB polygon feature class, an object ID in that feature class, its corresponding unique RCA ID, and a dataframe with three columns (OID, RCA ID, and Mean VB Width). In this way, two matching lists of object IDs and RCA IDs can be iterated through the function.

All temp files are written to the "in_memory" workspace. This was found to operate faster than the newer "memory" workspace. (See the ArcPro documentation here: https://pro.arcgis.com/en/pro-app/latest/help/analysis/geoprocessing/basics/the-in-memory-workspace.htm). If too many features are processed in memory during the for loop, the speed will decrease from 1-2 seconds per feature to 30-40 seconds per feature. The solution is to run this script individually for each clean valley bottom polygon, which will preserve the quick processing speed and clear the data debris from the in_memory workspace every time the individual script ends.

In [ ]:
# set up the valley bottom width calculation function
# arguments to the function are a RCA VB polygon feature class, an object ID in that feature class, 
# a corresponding unique RCA IDs in that feature class, and 
# a dataframe with three columns.... (OID, RCA, and MEAN WIDTH)

def vb_width(vbpoly, o, r, vb_width_df):
    
    #set uRCA ID variable for messaging
    uRCA = str(r)
    
    #set timestamped name for feature layer
    dtag = datetime.now().strftime("%Y%m%d%H%M%S")
    vb_ = "vb_" + dtag
    vb = vb_ + "_TEMP"
    vb = str("in_memory\\" + vb)

    #make feature layer from individual vb shape using a where clause
    where = str("OBJECTID = " + str(o))
    arcpy.management.MakeFeatureLayer(vbpoly, vb, where)


    #######################################################################################################


    #set timestamped name for temp polygon
    t_ = "mbr_" + dtag
    temp = t_ + "_TEMP"
    temp = str("in_memory\\" + temp)

    #create temporary polygon for minimum bounding rectangle
    arcpy.management.MinimumBoundingGeometry(vb, temp, "RECTANGLE_BY_WIDTH", "NONE", "", "MBG_FIELDS")

    #get orientation angle from first feature in minimum bounding polygon geometery attribute table
    with arcpy.da.SearchCursor(temp, ["MBG_Orientation"]) as geo_:
        for g_row in geo_:
            angle = g_row[0]

    # provide message
    msg = str("Minimum Bounding Rectangle Angle for Valley Bottom RCA " + uRCA + " is " + str(angle) + "...")
    print(msg)
    
    arcpy.AddMessage(msg)
    print(arcpy.GetMessages())


    #########################################################################################################


    #set timestamped name for temp polygon
    t2_ = "mbr_sub_" + dtag
    temp2 = t2_ + "_TEMP"
    temp2 = str("in_memory\\" + temp2)

    #subdivide minimum bounding rectangle temp polygon into 10 parts using the rectangle orientation angle
    arcpy.management.SubdividePolygon(temp, temp2, "NUMBER_OF_EQUAL_PARTS", 10, "", "", angle, "STRIPS")

    # provide message
    msg2 = str("Dividing " + uRCA + " into 10 equal parts...")
    print(msg2)
    
    arcpy.AddMessage(msg2)
    print(arcpy.GetMessages())


    ###########################################################################################################


    #set timestamped name for temp line
    t3_ = "mbr_sub_line_" + dtag
    temp3 = t3_ + "_TEMP"
    temp3 = str("in_memory\\" + temp3)

    #convert subdivided polygon into line features using the original valley bottom polygon as an intersecting feature
    arcpy.management.FeatureToLine([vb, temp2], temp3)

    # provide message
    msg3 = str("Converting into line features...")
    print(msg3)
    
    arcpy.AddMessage(msg3)
    print(arcpy.GetMessages())


    ############################################################################################################


    #select line features that fall within the original vb polygon
    #use a clementi selection to ignore lines that completely share the vb polygon boundary
    arcpy.management.SelectLayerByLocation(temp3, "WITHIN_CLEMENTINI", vb, "", "NEW_SELECTION", "")

    #search thru the selected features and write all the shape lengths to a list
    lengths = []

    with arcpy.da.SearchCursor(temp3, ["Shape_Length"]) as len_:
        for l_row in len_:
            l = l_row[0]
            lengths.append(l)

    #calculate the mean of the lengths
    avg_vb_width = statistics.mean(lengths)

    #provide a message
    msg4 = str("Average width of valley bottom line divisions in RCA " + uRCA + " is " + str(avg_vb_width) + " meters...")
    print(msg4)
    
    arcpy.AddMessage(msg4)
    print(arcpy.GetMessages())


    ############################################################################################################


    #append the OID, RCA ID, and mean valley bottom width to a list, and write to the dataframe

    new_row = [o, r, avg_vb_width]
    i = len(vb_width_df)
    vb_width_df.loc[i+1] = new_row

    #provide a message
    msg5 = str("Writing results to a dataframe...")
    print(msg5)
    
    arcpy.AddMessage(msg5)
    print(arcpy.GetMessages())
    

    #############################################################################################################

    #delete intermediary layers
    msg6 = str("Deleting intermediary layers...")
    print(msg6)
    
    arcpy.AddMessage(msg6)
    print(arcpy.GetMessages())
    
    
    # start with vb feature layer (not a feature class...)
    
    arcpy.management.Delete(vb)
    arcpy.management.Delete(temp)
    arcpy.management.Delete(temp2)
    arcpy.management.Delete(temp3)
    
    
    
    ##### IF NOT USING THE IN MEMORY WORKSPACE....
    ##### the workflow below can be used to delete feature classes from disk
    
    
    
    # then find all feature classes created in the MBR process...
    # list them both by their short environment variable names and by their full GDB paths
    # try deleting using both methods...
    
    #all_fcs = arcpy.ListFeatureClasses()
    
    #mbrs = []

    #for a in all_fcs:
        #if a.endswith("TEMP") ==  True:
            #mbrs.append(a)

    #for m in mbrs:

        #try:
            #arcpy.management.Delete(m)

        #except:
            
            #msg8 = str("Could not delete " + m + """ ...abort the script NOW to prevent 
            #overloading the project geodatabase and crashing the program!!!""")
            
            #print(msg8)

            #arcpy.AddMessage(msg8)
            #print(arcpy.GetMessages())

For each RCA VB feature class, create a list of OIDs and RCAs to use in the function. Also create an empty dataframe to hold function results. Supply all three of these parameters to the function and run it for each RCA VB feature class.

In [ ]:
for RCA_VB in RCA_VBs:
    
    #count valley bottom polygon features and start a counter
    vbcount = arcpy.management.GetCount(RCA_VB)

    count_msg1 = str("There are " + str(vbcount) + " valley bottom polygons to process...")
    print(count_msg1)
    arcpy.AddMessage(count_msg1)
    print(arcpy.GetMessages())

    counter = 1

    #create list of features to process and their RCA ids, and populate them
    uRCA_list = []
    oid_list = []

    with arcpy.da.SearchCursor(RCA_VB,['OBJECTID', rid]) as cursor:
        for row in cursor:

            oid = row[0]
            rca = row[1]

            oid_list.append(oid)
            uRCA_list.append(rca)

    #create a new empty dataframe to hold results
    vb_width_df = pandas.DataFrame(columns = ["OID", "RCA_VB", "MEAN_VB_WIDTH_M"])
    
    #loop thru all features for processing
    for o, r in zip(oid_list, uRCA_list):        

        #set uRCA ID variable for messaging
        uRCA = r

        #provide counter message
        count_msg2 = str("Processing valley bottom " + str(uRCA) + " (" + str(counter) + " of " + str(vbcount) + ")...")
        print(count_msg2)

        arcpy.AddMessage(count_msg2)
        print(arcpy.GetMessages())

        
        #advance counter for next feature
        counter = counter + 1

        ####################################################################################################

        #run function
        vb_width(RCA_VB, o, r, vb_width_df)

        ####################################################################################################


    # write the dataframe to CSV using the user-defined output directory... then re-import into the GDB
    i = RCA_VB.split("_")[-2]
    out = str(temp + "\RCA_VB_Width_" + g + "_" + str(i) + ".csv")
    out2 = str("RCA_VB_Width_" + str(i))
    
    vb_width_df.to_csv(out, index = False)
    
    arcpy.conversion.TableToTable(out, gdb, out2)